# 01 · Parse with `ai_parse_document` + Cost

Parse the corpus into a **materialized Delta table** with `ai_parse_document`, flatten the
layout-aware output to text, then query the **actual cost** from `system.billing`.

`ai_parse_document` emits a `VARIANT` of layout-aware elements (tables, headings, OCR for
image-only pages). We flatten `document.elements[].content` to one text column for
`ai_extract` in notebook `02`.

In [ ]:
CATALOG, SCHEMA, VOLUME = "users", "scott_mckean", "insurance_pdfs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
from time import perf_counter

## 1 · Parse → materialized `parsed_docs` / `parsed_text`

In [ ]:
t0 = perf_counter()
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.parsed_docs AS
    SELECT regexp_extract(path, '([^/]+)$', 1) AS filename,
           ai_parse_document(content)         AS parsed
    FROM read_files('{VOLUME_PATH}/real', format => 'binaryFile')
""")
parse_wall_s = perf_counter() - t0

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.parsed_text AS
    SELECT filename,
           CONCAT_WS('\n',
               TRANSFORM(
                   from_json(to_json(parsed:document:elements),
                             'ARRAY<STRUCT<content:STRING,type:STRING>>'),
                   x -> x.content)) AS full_text
    FROM {CATALOG}.{SCHEMA}.parsed_docs
""")
n = spark.table(f"{CATALOG}.{SCHEMA}.parsed_docs").count()
print(f"Parsed {n} docs in {parse_wall_s:.1f}s ({n/parse_wall_s:.2f} docs/s)")
display(spark.sql(f"""
    SELECT filename, substr(full_text, 1, 400) AS text_preview
    FROM {CATALOG}.{SCHEMA}.parsed_text LIMIT 3
"""))

## 2 · Cost of the parse run

`ai_parse_document`, `ai_extract`, `ai_classify` bill under the **`AI_FUNCTIONS`** product
in `system.billing.usage` (`ai_query` bills under `MODEL_SERVING`). `system.billing` can
lag a few hours, so we also show an a-priori estimate.

In [ ]:
n_docs = spark.table(f"{CATALOG}.{SCHEMA}.parsed_docs").count()
est_pages = n_docs * 2          # ACORD samples are ~1-3 pages
for rate in (1, 6):             # parse ~$1-6 / 1k pages by complexity
    print(f"  parse @ ${rate}/1k pages -> ${est_pages/1000*rate:.4f} for ~{est_pages:.0f} pages")
print("  combined parse+extract ~ $8.75 / 1k pages at medium complexity")
print("  (point read_files at a larger volume to scale this up)")

In [ ]:
cost = spark.sql("""
    WITH u AS (
        SELECT sku_name, cloud, max(usage_end_time) AS ute, sum(usage_quantity) AS dbus
        FROM system.billing.usage
        WHERE billing_origin_product = 'AI_FUNCTIONS'
          AND usage_date >= current_date() - INTERVAL 2 DAYS
        GROUP BY sku_name, cloud
    )
    SELECT u.sku_name, round(u.dbus, 3) AS dbus,
           round(u.dbus * p.pricing.default, 4) AS est_usd
    FROM u JOIN system.billing.list_prices p
      ON u.cloud = p.cloud AND u.sku_name = p.sku_name
     AND u.ute >= p.price_start_time
     AND (p.price_end_time IS NULL OR u.ute < p.price_end_time)
    ORDER BY dbus DESC
""")
if cost.count() == 0:
    print("No AI_FUNCTIONS usage recorded yet (billing lag) - use the estimate above.")
else:
    print("AI_FUNCTIONS spend, last 2 days:"); display(cost)

_Next: `02_extract_fields`._